# Ingestão Raw: Vendas

**Objetivo:** ler os arquivos JSON de vendas (particionados por país e data na origem) da Landing Zone, usando Databricks Autoloader, e gravar como tabela Delta na camada Raw.

**Fonte:** `/Volumes/poc_latam_food/landing/blob_simulado/vendas/` (o Autoloader lê recursivamente todas as subpastas `pais=X/data=Y/`)

**Destino:** tabela Delta `raw.vendas` (schema `raw`, catalog `poc_latam_food`), particionada por `data_ingestao` — conforme decisão de arquitetura, para viabilizar o expurgo por partição no notebook seguinte (`expurgo_raw.py`).

**Modo de execução:** batch (`trigger(availableNow=True)`).

**Checkpoint:** `/Volumes/poc_latam_food/landing/blob_simulado/_checkpoints/vendas/`.

**Observação de retenção:** esta tabela **está sujeita ao TTL de 48h** (diferente das dimensões) — dado seu alto volume e natureza transacional recorrente.

In [0]:
# Imports

from pyspark.sql.functions import current_timestamp

print("Imports realizados com sucesso.")

In [0]:
# Ingestão vendas

from pyspark.sql.functions import to_date

caminho_origem_vendas = "/Volumes/poc_latam_food/landing/blob_simulado/vendas"
caminho_checkpoint_vendas = "/Volumes/poc_latam_food/landing/blob_simulado/_checkpoints/vendas"
caminho_schema_vendas = "/Volumes/poc_latam_food/landing/blob_simulado/_schemas/vendas"

df_stream_vendas = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", caminho_schema_vendas)
    .load(caminho_origem_vendas)
    .withColumn("data_ingestao", current_timestamp())
    .withColumn("data_ingestao_particao", to_date(current_timestamp()))
)

(
    df_stream_vendas
    .writeStream
    .format("delta")
    .option("checkpointLocation", caminho_checkpoint_vendas)
    .partitionBy("data_ingestao_particao")
    .trigger(availableNow=True)
    .toTable("poc_latam_food.raw.vendas")
)

print("Ingestão de Vendas concluída.")

In [0]:
# Validação visual - raw.vendas

print("=== raw.vendas ===")
spark.table("poc_latam_food.raw.vendas").display()

print(f"Total de linhas: {spark.table('poc_latam_food.raw.vendas').count()}")

print("Contagem por país:")
spark.table("poc_latam_food.raw.vendas").groupBy("pais").count().display()

print("Contagem por partição (data_ingestao_particao):")
spark.table("poc_latam_food.raw.vendas").groupBy("data_ingestao_particao").count().display()